In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
import sys
import math
import logging
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
from pixell import reproject, lensing, enmap, utils

sys.path.append("/users/stevensonb/Research/tools/deepsphere-cosmo-tf2")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf

tf.get_logger().setLevel(logging.ERROR)

from tensorflow.keras.layers import Dropout, AlphaDropout
from tensorflow.keras.callbacks import EarlyStopping, TerminateOnNaN
from tensorflow.keras.optimizers import AdamW, Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay, CosineDecayRestarts

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import (
    HealpyChebyshev,
    HealpyPool,
    HealpyPseudoConv_Transpose,
    Healpy_Transformer,
)

from mlpng import Core
from mlpng.utils import setup_logging, RMSELoss, rmse_metrics
from mlpng.utils.dataloaders import KappaDataset, MapDataset
from mlpng.scn_jorik import get_model as get_fnl_model

logger = setup_logging("mlpng.notebook", level=logging.DEBUG)

In [ ]:
print("Conda environment:", os.environ["CONDA_DEFAULT_ENV"])
print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs: {len(gpus)}")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
core = Core(
    [
        "settings/n256.json",
        "--nsims",
        "10000",
        "--phi_scale",
        "1",
        "--shapes",
        "local",
        "--fnl_range",
        "-1000",
        "1000",
    ]
)
shapes = core.shapes

In [ ]:
batch_size = 32
max_epochs = 100
initial_LR = 1e-3

data_fraction = 0.01
unet_split = np.array([0.8, 0.1, 0.1]) * data_fraction
unet_duplicates = [25, 1, 1]

final_split = np.array([0.8, 0.1, 0.1]) * data_fraction
final_duplicates = [25, 1, 1]

strategy = tf.distribute.MirroredStrategy()

In [ ]:
date_time = tf.timestamp().numpy().astype(int)
run_name = f"notebook-{date_time}"
save_dir = f"{core.dirs['model']}/{core.name}"
run_info = f"{core.shapes_str}-{run_name}"

unet_keras_file = f"{save_dir}/unet-{run_info}.keras"
fnl_keras_file = f"{save_dir}/fnl-{run_info}.keras"

unet_cache = f"{core.name}/unet-{core.name}-f{data_fraction}-k100"  # New cache name with kappa_scale indicator

os.makedirs(save_dir, exist_ok=True)
print(f"Run name: {run_name}")
print(f"U-Net file: {unet_keras_file}")

In [ ]:
callbacks = [
    TerminateOnNaN(),
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
]

In [ ]:
@tf.keras.saving.register_keras_serializable()
class EncoderBlock(tf.keras.layers.Layer):
    def __init__(
        self,
        nside,
        npix,
        fin,
        fout,
        activation,
        max_batch_size,
        K,
        use_bn=True,
        use_bias=False,
        pool=True,
        dropout_rate=0.1,
        initializer=None,
        use_alpha_dropout=False,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.nside = nside
        self.npix = npix
        self.fin = fin
        self.fout = fout
        self.activation_fn = activation
        self.max_batch_size = max_batch_size
        self.K = K
        self.use_bn = use_bn
        self.use_bias = use_bias
        self.dropout_rate = dropout_rate
        self.pool = pool
        self.initializer = initializer
        self.use_alpha_dropout = use_alpha_dropout

        enc_layers = [
            # HealpyChebyshev(
            #     K=K,
            #     Fout=fout,
            #     activation=None,
            #     use_bn=False,
            #     use_bias=False,
            #     initializer=initializer,
            # ),
            HealpyChebyshev(
                K=K,
                Fout=fout,
                activation=activation,
                use_bn=use_bn,
                use_bias=use_bias,
                initializer=initializer,
            ),
        ]
        if dropout_rate > 0.0:
            if use_alpha_dropout:
                enc_layers.append(AlphaDropout(dropout_rate))
            else:
                enc_layers.append(Dropout(dropout_rate))

        self.body = HealpyGCNN(
            nside=nside,
            indices=np.arange(npix),
            layers=enc_layers,
            n_neighbors=8,
            max_batch_size=max_batch_size,
            initial_Fin=fin,
        )

        self.pooler = None
        if pool:
            self.pooler = HealpyGCNN(
                nside=nside,
                indices=np.arange(npix),
                layers=[HealpyPool(2, "AVG")],  # p=2 for 16x reduction (4^2)
                n_neighbors=8,
                max_batch_size=max_batch_size,
                initial_Fin=fout,  # Fixed: pooler receives output from body which has fout channels
            )

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "nside": self.nside,
                "npix": self.npix,
                "fin": self.fin,
                "fout": self.fout,
                "activation": self.activation_fn,
                "max_batch_size": self.max_batch_size,
                "K": self.K,
                "use_bn": self.use_bn,
                "use_bias": self.use_bias,
                "pool": self.pool,
                "dropout_rate": self.dropout_rate,
                "initializer": (
                    tf.keras.initializers.serialize(self.initializer)
                    if self.initializer
                    else None
                ),
                "use_alpha_dropout": self.use_alpha_dropout,
            }
        )
        return config

    @classmethod
    def from_config(cls, config):
        config = config.copy()
        if config.get("initializer"):
            config["initializer"] = tf.keras.initializers.deserialize(
                config["initializer"]
            )
        return cls(**config)

    def call(self, inputs, training=False):
        x = skip = self.body(inputs, training=training)
        if self.pool:
            x = self.pooler(x, training=training)
        return x, skip

In [ ]:
@tf.keras.saving.register_keras_serializable()
class DecoderBlock(tf.keras.layers.Layer):
    def __init__(
        self,
        nside,
        npix,
        fin,
        fout,
        activation,
        K,
        max_batch_size,
        skip_channels=0,  # Number of channels in skip connection (0 = no skip)
        upsample=True,
        use_bn=True,
        use_bias=False,
        dropout_rate=0.1,
        initializer=None,
        use_alpha_dropout=False,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.nside = nside
        self.npix = npix
        self.fin = fin
        self.fout = fout
        self.activation_fn = activation
        self.max_batch_size = max_batch_size
        self.K = K
        self.use_bn = use_bn
        self.use_bias = use_bias
        self.dropout_rate = dropout_rate
        self.upsample = upsample
        self.skip_channels = skip_channels
        self.initializer = initializer
        self.use_alpha_dropout = use_alpha_dropout

        # Calculate input resolution (before upsampling)
        # With p=2 upsampling: 16x spatial increase (4^2)
        if upsample:
            input_npix = npix // 16  # 16x reduction for p=2
            input_nside = nside // 4  # nside reduces by 4 for p=2
        else:
            input_npix = npix
            input_nside = nside

        # Upsampling layers - operates at input resolution, outputs to npix resolution
        self.upsampler = None
        if upsample:
            upsample_layers = [
                HealpyPseudoConv_Transpose(
                    2, fout, kernel_initializer=initializer
                )  # p=2 for 16x upsampling
            ]
            self.upsampler = HealpyGCNN(
                nside=input_nside,
                indices=np.arange(input_npix),
                layers=upsample_layers,
                n_neighbors=8,
                max_batch_size=max_batch_size,
                initial_Fin=fin,
            )
        # Convolution after addition - operates at output resolution (npix)
        # With additive skips, input channels = fout (skip channels must match fout)
        conv_input_channels = fout if upsample else fin
        conv_layers = [
            # HealpyChebyshev(
            #     K=K,
            #     Fout=fout,
            #     activation=None,
            #     use_bn=False,
            #     use_bias=False,
            #     initializer=initializer,
            # ),
            HealpyChebyshev(
                K=K,
                Fout=fout,
                activation=activation,
                use_bn=use_bn,
                use_bias=use_bias,
                initializer=initializer,
            ),
        ]
        if dropout_rate > 0.0:
            if use_alpha_dropout:
                conv_layers.append(AlphaDropout(dropout_rate))
            else:
                conv_layers.append(Dropout(dropout_rate))

        self.conv = HealpyGCNN(
            nside=nside,  # Output resolution
            indices=np.arange(npix),
            layers=conv_layers,
            n_neighbors=8,
            max_batch_size=max_batch_size,
            initial_Fin=conv_input_channels,
        )

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "nside": self.nside,
                "npix": self.npix,
                "fin": self.fin,
                "fout": self.fout,
                "activation": self.activation_fn,
                "K": self.K,
                "max_batch_size": self.max_batch_size,
                "skip_channels": self.skip_channels,
                "upsample": self.upsample,
                "use_bn": self.use_bn,
                "use_bias": self.use_bias,
                "dropout_rate": self.dropout_rate,
                "initializer": (
                    tf.keras.initializers.serialize(self.initializer)
                    if self.initializer
                    else None
                ),
                "use_alpha_dropout": self.use_alpha_dropout,
            }
        )
        return config

    @classmethod
    def from_config(cls, config):
        config = config.copy()
        if config.get("initializer"):
            config["initializer"] = tf.keras.initializers.deserialize(
                config["initializer"]
            )
        return cls(**config)

    def call(self, inputs, skip=None, training=False):
        # Upsample first (from input resolution to output resolution)
        if self.upsampler is not None:
            x = self.upsampler(inputs, training=training)
        else:
            x = inputs

        # Concatenate skip connection along channel axis
        if skip is not None:
            # Add skip connection (channels must match)

            x = x + skip
        x = self.conv(x, training=training)
        return x
        # Apply convolution at output resolution

In [ ]:
@tf.keras.saving.register_keras_serializable()
class ResidualHealpyUNet:
    def __init__(
        self,
        input_shape,
        activation="relu",
        max_batch_size=32,
        dropout_rate=0.1,
        use_bn=True,
        use_bias=True,
        initializer=None,
    ):
        self.input_shape = input_shape
        self.activation = activation
        self.max_batch_size = max_batch_size
        self.npix = input_shape[1]
        self.nside = hp.npix2nside(self.npix)
        self.npol = input_shape[2]
        self.dropout_rate = dropout_rate
        self.use_bias = use_bias
        self.initializer = initializer

        # Auto-detect activation configuration
        self.use_selu = activation == "selu" or (
            callable(activation) and getattr(activation, "__name__", "") == "selu"
        )
        self.use_gelu = activation == "gelu" or (
            callable(activation) and getattr(activation, "__name__", "") == "gelu"
        )

        if self.use_selu:
            self.use_bn = False  # SELU is self-normalizing, doesn't need BN
            self.use_alpha_dropout = True
            if initializer is None:
                self.initializer = tf.keras.initializers.LecunNormal()
            print(
                "SELU mode: BatchNorm disabled, using AlphaDropout and LecunNormal initializer"
            )
        elif self.use_gelu:
            self.use_bn = use_bn
            self.use_alpha_dropout = False
            if initializer is None:
                self.initializer = tf.keras.initializers.HeNormal()
            print("GELU mode: Using HeNormal initializer")
        else:
            self.use_bn = use_bn
            self.use_alpha_dropout = False

    def get_config(self):
        return {
            "input_shape": self.input_shape,
            "activation": self.activation,
            "max_batch_size": self.max_batch_size,
            "dropout_rate": self.dropout_rate,
            "use_bn": self.use_bn,
            "use_bias": self.use_bias,
            "initializer": (
                tf.keras.initializers.serialize(self.initializer)
                if self.initializer
                else None
            ),
        }

    def get_model(self):
        # Depth calculation for 16x pooling per level (p=2)
        # nside=64 -> depth=1 (bottleneck at nside=4)
        # nside=256 -> depth=2 (bottleneck at nside=4)
        depth = int(math.log(self.nside, 4)) - 1
        depth = max(1, depth)  # Ensure at least 1 level

        # Reduced channels by power of 2 (2^(i+6) instead of 2^(i+7))
        # For additive skips: encoder outputs same channels as decoder expects
        base_channels = [self.npol] + [2 ** (i + 6) for i in range(depth + 1)]

        # 16x reduction per level (4^2 = 16)
        level_npixels = [self.npix // (16**i) for i in range(depth + 1)]
        level_nsides = [hp.npix2nside(npix) for npix in level_npixels]
        Ks = [1 + (2 * (1 + i)) for i in range(depth + 1)]
        Ks = list(reversed(Ks))

        print(f"Channels: {base_channels}, K values: {Ks}")
        print(f"Level npixels: {level_npixels}")
        print(f"Level nsides: {level_nsides}")
        print(f"Depth: {depth}, Bottleneck nside: {level_nsides[-1]}")

        inputs = tf.keras.Input(shape=self.input_shape[1:], name="lensed")
        x = inputs
        skips = []

        # Encoder: progressively downsample with 16x pooling
        # Skip channels aligned with decoder output channels for additive skips
        for i in range(depth):
            # fout matches what decoder will output at this level for additive skip
            block = EncoderBlock(
                level_nsides[i],
                level_npixels[i],
                fin=base_channels[i],
                fout=base_channels[i + 1],  # Skip has these channels
                activation=self.activation,
                max_batch_size=self.max_batch_size,
                K=Ks[i],
                dropout_rate=self.dropout_rate,
                use_bias=self.use_bias,
                use_bn=self.use_bn,
                initializer=self.initializer,
                use_alpha_dropout=self.use_alpha_dropout,
            )
            x, skip = block(x)
            skips.append(skip)

        # Bottleneck at deepest level
        bottleneck = HealpyGCNN(
            nside=level_nsides[depth],
            indices=np.arange(level_npixels[depth]),
            layers=[
                HealpyChebyshev(
                    K=Ks[-1],
                    Fout=base_channels[-1],
                    activation=self.activation,
                    use_bias=self.use_bias,
                    use_bn=self.use_bn,
                    initializer=self.initializer,
                )
            ],
            n_neighbors=8,
            max_batch_size=self.max_batch_size,
            initial_Fin=base_channels[depth],
            name="bottleneck",
        )
        x = bottleneck(x)

        # Decoder: progressively upsample with 16x upsampling
        # fout must match skip channels for additive connection
        for i in reversed(range(1, depth + 1)):
            output_nside = level_nsides[i - 1]
            output_npix = level_npixels[i - 1]

            # First decoder iteration: input from bottleneck has base_channels[depth]
            # Subsequent iterations: input from previous decoder has base_channels[i]
            if i == depth:
                fin = base_channels[depth]  # From bottleneck
            else:
                fin = base_channels[i]  # From previous decoder output

            block = DecoderBlock(
                output_nside,
                output_npix,
                fin=fin,
                fout=base_channels[i],  # Output matches skip connection
                activation=self.activation,
                max_batch_size=self.max_batch_size,
                K=Ks[i - 1],
                skip_channels=0,  # Not used with additive skips
                dropout_rate=self.dropout_rate,
                use_bias=self.use_bias,
                use_bn=self.use_bn,
                initializer=self.initializer,
                use_alpha_dropout=self.use_alpha_dropout,
            )
            x = block(x, skips[i - 1])

        # Final upsampling: p=1 gives 4x pixel upsampling (2x nside)
        # Input: nside=64 (49152 px) -> Output: nside=128 (196608 px)
        output_nside = level_nsides[0] * 2
        output_npix = level_npixels[0] * 4

        output_layers = [
            HealpyPseudoConv_Transpose(
                1, base_channels[1] // 2, kernel_initializer=self.initializer
            ),
            HealpyChebyshev(
                K=1,
                Fout=base_channels[0],
                activation=None,
                use_bn=False,
                use_bias=True,
                initializer=self.initializer,
            ),
        ]
        output_head = HealpyGCNN(
            nside=level_nsides[0],
            indices=np.arange(level_npixels[0]),
            layers=output_layers,
            n_neighbors=8,
            max_batch_size=self.max_batch_size,
            initial_Fin=base_channels[1],
            name="kappa",
        )

        outputs = output_head(x)
        return tf.keras.Model(inputs, outputs)

In [ ]:
@tf.keras.saving.register_keras_serializable()
class TransformerHealpyUNet:
    """
    U-Net with Transformer bottleneck for global attention on spherical data.

    Uses HealpyChebyshev for encoder/decoder (local graph convolutions) and
    Healpy_Transformer in the bottleneck for global attention at lowest resolution.

    Design decisions:
    - Memory: Transformer only at bottleneck (nside=4 → 192 tokens) for efficiency
    - Channels: Projection layers before/after transformer to match channel dimensions
    - Activations: SELU for Chebyshev layers, GELU for Transformer (standard practice)
    """

    def __init__(
        self,
        input_shape,
        activation="relu",
        max_batch_size=32,
        dropout_rate=0.1,
        use_bn=True,
        use_bias=True,
        initializer=None,
        # Transformer-specific parameters
        num_heads=8,
        transformer_layers=2,
        key_dim=None,  # Auto-computed as base_channels[-1] // num_heads if None
    ):
        self.input_shape = input_shape
        self.activation = activation
        self.max_batch_size = max_batch_size
        self.npix = input_shape[1]
        self.nside = hp.npix2nside(self.npix)
        self.npol = input_shape[2]
        self.dropout_rate = dropout_rate
        self.use_bias = use_bias
        self.initializer = initializer

        # Transformer config
        self.num_heads = num_heads
        self.transformer_layers = transformer_layers
        self.key_dim = key_dim

        # Auto-detect activation configuration for Chebyshev layers
        self.use_selu = activation == "selu" or (
            callable(activation) and getattr(activation, "__name__", "") == "selu"
        )
        self.use_gelu = activation == "gelu" or (
            callable(activation) and getattr(activation, "__name__", "") == "gelu"
        )

        if self.use_selu:
            self.use_bn = False  # SELU is self-normalizing
            self.use_alpha_dropout = True
            if initializer is None:
                self.initializer = tf.keras.initializers.LecunNormal()
            print(
                "SELU mode: BatchNorm disabled, using AlphaDropout and LecunNormal initializer"
            )
        elif self.use_gelu:
            self.use_bn = use_bn
            self.use_alpha_dropout = False
            if initializer is None:
                self.initializer = tf.keras.initializers.HeNormal()
            print("GELU mode: Using HeNormal initializer")
        else:
            self.use_bn = use_bn
            self.use_alpha_dropout = False

    def get_config(self):
        return {
            "input_shape": self.input_shape,
            "activation": self.activation,
            "max_batch_size": self.max_batch_size,
            "dropout_rate": self.dropout_rate,
            "use_bn": self.use_bn,
            "use_bias": self.use_bias,
            "initializer": (
                tf.keras.initializers.serialize(self.initializer)
                if self.initializer
                else None
            ),
            "num_heads": self.num_heads,
            "transformer_layers": self.transformer_layers,
            "key_dim": self.key_dim,
        }

    def get_model(self):
        # Depth calculation for 16x pooling per level (p=2)
        # nside=64 -> depth=1 (bottleneck at nside=4)
        # nside=256 -> depth=2 (bottleneck at nside=4)
        depth = int(math.log(self.nside, 4)) - 1
        depth = max(1, depth)  # Ensure at least 1 level

        # Reduced channels by power of 2 (2^(i+6) instead of 2^(i+7))
        # For additive skips: encoder outputs same channels as decoder expects
        base_channels = [self.npol] + [2 ** (i + 5) for i in range(depth + 1)]

        # 16x reduction per level (4^2 = 16)
        level_npixels = [self.npix // (16**i) for i in range(depth + 1)]
        level_nsides = [hp.npix2nside(npix) for npix in level_npixels]
        Ks = [1 + (2 * (1 + i*0)) for i in range(depth + 1)]
        Ks = list(reversed(Ks))

        # Auto-compute key_dim to match channel dimensions
        bottleneck_channels = base_channels[-1]
        key_dim = (
            self.key_dim if self.key_dim else bottleneck_channels // self.num_heads
        )
        transformer_embed_dim = key_dim * self.num_heads

        print(f"Channels: {base_channels}, K values: {Ks}")
        print(f"Level npixels: {level_npixels}")
        print(f"Level nsides: {level_nsides}")
        print(f"Depth: {depth}, Bottleneck nside: {level_nsides[-1]}")
        print(
            f"Transformer config: {self.num_heads} heads, key_dim={key_dim}, embed_dim={transformer_embed_dim}"
        )
        print(
            f"Bottleneck tokens: {level_npixels[depth]} (nside={level_nsides[depth]})"
        )

        inputs = tf.keras.Input(shape=self.input_shape[1:], name="lensed")
        x = inputs
        skips = []

        # Encoder: progressively downsample with 16x pooling
        # Skip channels aligned with decoder output channels for additive skips
        for i in range(depth):
            # fout matches what decoder will output at this level for additive skip
            block = EncoderBlock(
                level_nsides[i],
                level_npixels[i],
                fin=base_channels[i],
                fout=base_channels[i + 1],  # Skip has these channels
                activation=self.activation,
                max_batch_size=self.max_batch_size,
                K=Ks[i],
                dropout_rate=self.dropout_rate,
                use_bias=self.use_bias,
                use_bn=self.use_bn,
                initializer=self.initializer,
                use_alpha_dropout=self.use_alpha_dropout,
            )
            x, skip = block(x)
            skips.append(skip)

        # Transformer Bottleneck at deepest level
        # 1. Project to transformer embedding dimension if needed
        # 2. Apply Healpy_Transformer with sparse graph attention
        # 3. Project back to bottleneck channels

        bottleneck_layers = []

        # Input projection if channel mismatch
        if base_channels[depth] != transformer_embed_dim:
            bottleneck_layers.append(
                HealpyChebyshev(
                    K=Ks[-1],
                    Fout=transformer_embed_dim,
                    activation=None,  # Linear projection
                    use_bias=True,
                    use_bn=False,
                    initializer=self.initializer,
                )
            )

        # Transformer attention (uses GELU internally, LayerNorm handled by transformer)
        bottleneck_layers.append(
            Healpy_Transformer(
                key_dim=key_dim,
                num_heads=self.num_heads,
                positional_encoding=True,
                n_layers=self.transformer_layers,
                activation="gelu",  # Standard for transformers
                layer_norm=True,
            )
        )

        # Output projection back to expected channels
        bottleneck_layers.append(
            HealpyChebyshev(
                K=Ks[-1],
                Fout=bottleneck_channels,
                activation=self.activation,  # Use main activation here
                use_bias=self.use_bias,
                use_bn=self.use_bn,
                initializer=self.initializer,
            )
        )

        bottleneck = HealpyGCNN(
            nside=level_nsides[depth],
            indices=np.arange(level_npixels[depth]),
            layers=bottleneck_layers,
            n_neighbors=8,
            max_batch_size=self.max_batch_size,
            initial_Fin=base_channels[depth],
            name="transformer_bottleneck",
        )
        x = bottleneck(x)

        # Decoder: progressively upsample with 16x upsampling
        # fout must match skip channels for additive connection
        for i in reversed(range(1, depth + 1)):
            output_nside = level_nsides[i - 1]
            output_npix = level_npixels[i - 1]

            # First decoder iteration: input from bottleneck has base_channels[depth]
            # Subsequent iterations: input from previous decoder has base_channels[i]
            if i == depth:
                fin = base_channels[depth]  # From bottleneck
            else:
                fin = base_channels[i]  # From previous decoder output

            block = DecoderBlock(
                output_nside,
                output_npix,
                fin=fin,
                fout=base_channels[i],  # Output matches skip connection
                activation=self.activation,
                max_batch_size=self.max_batch_size,
                K=Ks[i - 1],
                skip_channels=0,  # Not used with additive skips
                dropout_rate=self.dropout_rate,
                use_bias=self.use_bias,
                use_bn=self.use_bn,
                initializer=self.initializer,
                use_alpha_dropout=self.use_alpha_dropout,
            )
            x = block(x, skips[i - 1])

        # Final upsampling: p=1 gives 4x pixel upsampling (2x nside)
        # Input: nside=64 (49152 px) -> Output: nside=128 (196608 px)
        output_nside = level_nsides[0] * 2
        output_npix = level_npixels[0] * 4

        output_layers = [
            HealpyPseudoConv_Transpose(
                1, base_channels[1] // 2, kernel_initializer=self.initializer
            ),
            HealpyChebyshev(
                K=1,
                Fout=base_channels[0],
                activation=None,
                use_bn=False,
                use_bias=True,
                initializer=self.initializer,
            ),
        ]
        output_head = HealpyGCNN(
            nside=level_nsides[0],
            indices=np.arange(level_npixels[0]),
            layers=output_layers,
            n_neighbors=8,
            max_batch_size=self.max_batch_size,
            initial_Fin=base_channels[1],
            name="kappa",
        )

        outputs = output_head(x)
        return tf.keras.Model(inputs, outputs)

In [ ]:
logger.info("Creating U-Net model and training data")
epoch_steps = math.ceil(
    core.total_sims * unet_split[0] * unet_duplicates[0] // batch_size
)
decay_steps = epoch_steps  # * 2

ds = KappaDataset.fromCore(
    core,
    phi_scale=1,
    kappa_scale=1,  # Reduced from sqrt(1e7) to match input scale better
    x_output="lensed",
    y_output="kappa",
)

train, val, test = ds.split(
    train_size=unet_split[0],
    val_size=unet_split[1],
    test_size=unet_split[2],
    to_tf=True,
    batch_size=batch_size,
    duplicates=unet_duplicates,
    buffer_size=len(ds),
    # cache=False,
    rotate=[True, False, False],  # Enable rotation for training data
    gen_batch_size=core.slurm.n_cpus * 8,
)

In [ ]:
with strategy.scope():
    # CosineDecayRestarts: first_decay_steps = 2 epochs, then restart with alpha=0.01 minimum
    learning_rate = CosineDecayRestarts(
        initial_learning_rate=1e-3,
        first_decay_steps=decay_steps,
        t_mul=2.0,  # Each restart period is 2x longer
        m_mul=0.95,  # LR multiplier after each restart
        alpha=0.01,  # Minimum LR as fraction of initial
    )

    # GELU activation with He Normal initialization
    u_net = ResidualHealpyUNet(
        (None, core.npix, core.npols),
        activation="tanh",
        max_batch_size=batch_size,
        dropout_rate=0.1,
    ).get_model()
    u_net.compile(
        optimizer=AdamW(1e-4), loss="mse", metrics=["mae", "accuracy"]
    )

u_net.summary()

In [ ]:
logger.info("Training U-Net")
history = u_net.fit(
    train, epochs=100, validation_data=val, callbacks=callbacks, verbose=1
)

# u_net.save(unet_keras_file)
# logger.info(f"U-Net saved to {unet_keras_file}")

In [ ]:
if "history" in locals():
    plt.figure(figsize=(10, 4))
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss (MSE)")
    plt.title("U-Net Training Loss")
    plt.yscale("log")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Delens lensed maps and train fnl model

In [ ]:
res_arcmin = hp.nside2resol(core.nside, arcmin=True)
res_rad = res_arcmin * utils.arcmin
ny = int(round(np.pi / res_rad))
res_fixed = np.pi / ny
shape, wcs = enmap.fullsky_geometry(res_fixed)


def _delens_py(lensed, phi, nside=core.nside):
    """Delens maps using phi maps (already converted from kappa via kappa_to_phi)"""
    delensed_maps = np.zeros_like(lensed)
    for i, (l_map, p_map) in enumerate(zip(lensed, phi)):
        l_map = hp.reorder(l_map.numpy().flatten(), n2r=True).astype(np.float32)
        p_map = hp.reorder(p_map.numpy().flatten(), n2r=True).astype(np.float32)

        lensed_enmap = reproject.healpix2map(l_map, shape, wcs)
        phi_enmap = reproject.healpix2map(p_map, shape, wcs)
        delensed_enmap = lensing.delens_map(lensed_enmap, phi_enmap)

        m = reproject.map2healpix(delensed_enmap, nside)
        m = hp.remove_dipole(m, copy=False)
        delensed_maps[i] = m[:, None]

    return delensed_maps


def _map_fn(pair, phi):
    lensed, fnls = pair
    delensed = tf.py_function(func=_delens_py, inp=[lensed, phi], Tout=tf.float32)
    delensed.set_shape([None, core.npix, core.npols])
    fnls.set_shape([None, len(shapes)])
    return delensed, fnls

In [ ]:
logger.info("Creating delensed dataset for fnl training")
ds_lensed = MapDataset.fromCore(core, lensed=True)
train_3, val_3, test_3 = ds_lensed.split(
    train_size=final_split[0],
    val_size=final_split[1],
    test_size=final_split[2],
    to_tf=True,
    batch_size=batch_size,
    duplicates=final_duplicates,
    gen_batch_size=core.slurm.n_cpus,
)

fnl_truth = np.concatenate([y for _, y in test_3])

# Get kappa predictions and delens
logger.info("Predicting kappa with U-Net and delensing maps")
kappa_preds = u_net.predict(train_3, verbose=0)
kappa_preds_delens = ds.kappa_to_phi(kappa_preds)
kappa_ds = tf.data.Dataset.from_tensor_slices(kappa_preds_delens).batch(batch_size)

delensed_train = (
    tf.data.Dataset.zip((train_3, kappa_ds))
    .map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

# Same for validation
kappa_preds_val = u_net.predict(val_3, verbose=0)
kappa_preds_val_delens = ds.kappa_to_phi(kappa_preds_val)
kappa_ds_val = tf.data.Dataset.from_tensor_slices(kappa_preds_val_delens).batch(
    batch_size
)
delensed_val = (
    tf.data.Dataset.zip((val_3, kappa_ds_val))
    .map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

# And for test
kappa_preds_test = u_net.predict(test_3, verbose=0)
kappa_preds_test_delens = ds.kappa_to_phi(kappa_preds_test)
kappa_ds_test = tf.data.Dataset.from_tensor_slices(kappa_preds_test_delens).batch(
    batch_size
)
delensed_test = (
    tf.data.Dataset.zip((test_3, kappa_ds_test))
    .map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
logger.info("Creating and training fnl model on delensed maps")
epoch_steps = math.ceil(
    core.total_sims * final_split[0] * final_duplicates[0] // batch_size
)
decay_steps = epoch_steps * 3

with strategy.scope():
    learning_rate = ExponentialDecay(initial_LR, decay_steps, 0.95, staircase=True)
    fnl_model = get_fnl_model((None, core.npix, core.npols), batch_size, len(shapes))
    fnl_model.compile(
        optimizer=AdamW(learning_rate), loss=RMSELoss(), metrics=rmse_metrics(shapes)
    )

logger.info("Training fnl model")
fnl_history = fnl_model.fit(
    delensed_train,
    epochs=10,  # max_epochs,
    validation_data=delensed_val,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(fnl_history.history["loss"], label="Train Loss")
plt.plot(fnl_history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("fnl Model Training Loss")
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## fnl Predictions on Delensed Test Set

In [ ]:
logger.info("Predicting fnl on delensed test set")
fnl_preds = fnl_model.predict(delensed_test, verbose=0)

fnl_truth_flat = fnl_truth.ravel()
fnl_preds_flat = fnl_preds.ravel()

print(f"True fnls shape: {fnl_truth_flat.shape}")
print(f"Predicted fnls shape: {fnl_preds_flat.shape}")

fnl_error = fnl_preds_flat - fnl_truth_flat
print(f"\nMetrics:")
print(f"Mean absolute error: {np.mean(np.abs(fnl_error)):.4f}")
print(f"RMSE: {np.sqrt(np.mean(fnl_error**2)):.4f}")

sigma = core.get_likelihoods(True)[0]
print(f"Sigma (likelihood bound): {sigma:.4f}")

In [ ]:
line = np.array([np.nanmin(fnl_truth_flat), np.nanmax(fnl_truth_flat)])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Scatter plot
axes[0].scatter(fnl_truth_flat, fnl_preds_flat, alpha=0.5, s=10)
axes[0].plot(line, line, "r--", label="Perfect prediction")
axes[0].plot(line, line + sigma, "g--", label=f"+σ={sigma:.1f}")
axes[0].plot(line, line - sigma, "g--", label=f"-σ={-sigma:.1f}")
axes[0].set_xlabel("True fnl")
axes[0].set_ylabel("Predicted fnl")
axes[0].set_title("fnl Prediction from Delensed Maps")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error histogram
axes[1].hist(fnl_error, bins=50, color="C0", alpha=0.8, edgecolor="black")
axes[1].axvline(sigma, color="g", linestyle="--", linewidth=2, label=f"+σ={sigma:.1f}")
axes[1].axvline(
    -sigma, color="g", linestyle="--", linewidth=2, label=f"-σ={-sigma:.1f}"
)
axes[1].set_xlabel("Prediction Error")
axes[1].set_ylabel("Count")
axes[1].set_title("fnl Error Distribution")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Layer Activation Visualization

In [ ]:
# Visualize activations at each layer of the U-Net
# Get a sample from the test dataset
for x_sample, y_sample in test.take(1):
    test_input = x_sample[:1]  # Take first sample
    test_target = y_sample[:1]
    break

print(f"Input shape: {test_input.shape}")
print(f"Target shape: {test_target.shape}")

# For models with custom nested layers (EncoderBlock, DecoderBlock),
# we manually run through the model to extract activations at each stage

print("\nExtracting activations by running through model layers...")

layer_names = []
activations = []

x = test_input
skips = []  # Store skip connections for decoder blocks

for layer in u_net.layers:
    if isinstance(layer, tf.keras.layers.InputLayer):
        layer_names.append("input")
        activations.append(x.numpy() if hasattr(x, "numpy") else x)
        continue

    layer_name = layer.name

    # Handle EncoderBlock (returns tuple: output, skip)
    if "encoder_block" in layer_name.lower():
        x, skip = layer(x, training=False)
        skips.append(skip)
        # Store both output and skip
        layer_names.append(f"{layer_name}_out")
        activations.append(x.numpy())
        layer_names.append(f"{layer_name}_skip")
        activations.append(skip.numpy())

    # Handle DecoderBlock (needs skip connection)
    elif "decoder_block" in layer_name.lower():
        # Decoder blocks use skip connections in reverse order
        if len(skips) > 0:
            skip = skips.pop()
            x = layer(x, skip=skip, training=False)
        else:
            x = layer(x, skip=None, training=False)
        layer_names.append(layer_name)
        activations.append(x.numpy())

    # Handle standard layers (HealpyGCNN bottleneck, kappa head)
    else:
        try:
            x = layer(x, training=False)
            layer_names.append(layer_name)
            if hasattr(x, "numpy"):
                activations.append(x.numpy())
            else:
                activations.append(np.array(x))
        except Exception as e:
            print(f"  Skipping {layer_name}: {e}")

print(f"\nFound {len(activations)} layer activations:")
for name, act in zip(layer_names, activations):
    print(
        f"  {name}: shape={act.shape}, min={act.min():.4f}, max={act.max():.4f}, mean={act.mean():.4f}"
    )

In [ ]:
# Plot activation statistics across layers
if len(activations) >= 2:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Activation magnitude distribution across layers
    ax = axes[0, 0]
    means = [act.mean() for act in activations]
    stds = [act.std() for act in activations]
    mins = [act.min() for act in activations]
    maxs = [act.max() for act in activations]

    x_pos = np.arange(len(layer_names))
    ax.fill_between(x_pos, mins, maxs, alpha=0.3, label="Min-Max range")
    ax.plot(x_pos, means, "b-", linewidth=2, label="Mean")
    ax.fill_between(
        x_pos,
        np.array(means) - np.array(stds),
        np.array(means) + np.array(stds),
        alpha=0.3,
        color="blue",
        label="±1 Std",
    )
    ax.set_xlabel("Layer Index")
    ax.set_ylabel("Activation Value")
    ax.set_title("Activation Statistics Across Layers")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(layer_names, rotation=45, ha="right", fontsize=8)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 2. Activation variance (looking for dead/saturated layers)
    ax = axes[0, 1]
    variances = [act.var() for act in activations]
    ax.bar(x_pos, variances, alpha=0.7)
    ax.set_xlabel("Layer Index")
    ax.set_ylabel("Variance")
    ax.set_title("Activation Variance per Layer")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(layer_names, rotation=45, ha="right", fontsize=8)
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3, axis="y")

    # 3. Histogram of activations for selected layers
    ax = axes[1, 0]
    n_layers = len(activations)
    # Select up to 5 layers evenly spaced
    if n_layers <= 5:
        selected_indices = list(range(n_layers))
    else:
        selected_indices = [
            0,
            n_layers // 4,
            n_layers // 2,
            3 * n_layers // 4,
            n_layers - 1,
        ]
    colors = plt.cm.viridis(np.linspace(0, 1, len(selected_indices)))

    for idx, color in zip(selected_indices, colors):
        act = activations[idx].flatten()
        # Sample if too many values
        if len(act) > 10000:
            act = np.random.choice(act, 10000, replace=False)
        ax.hist(
            act,
            bins=50,
            alpha=0.5,
            color=color,
            label=f"{layer_names[idx]}",
            density=True,
        )

    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Density")
    ax.set_title("Activation Distribution (Selected Layers)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # 4. Percentage of "dead" (near-zero) activations per layer
    ax = axes[1, 1]
    dead_threshold = 1e-6
    dead_percentages = [
        (np.abs(act) < dead_threshold).mean() * 100 for act in activations
    ]
    saturated_threshold = 0.99  # For sigmoid activations
    saturated_percentages = [
        (np.abs(act) > saturated_threshold).mean() * 100 for act in activations
    ]

    width = 0.35
    ax.bar(
        x_pos - width / 2,
        dead_percentages,
        width,
        label=f"Dead (<{dead_threshold})",
        alpha=0.7,
    )
    ax.bar(
        x_pos + width / 2,
        saturated_percentages,
        width,
        label=f"Saturated (>{saturated_threshold})",
        alpha=0.7,
    )
    ax.set_xlabel("Layer Index")
    ax.set_ylabel("Percentage (%)")
    ax.set_title("Dead/Saturated Neurons per Layer")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(layer_names, rotation=45, ha="right", fontsize=8)
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

    # Print summary warnings
    print("\n" + "=" * 60)
    print("ACTIVATION HEALTH SUMMARY")
    print("=" * 60)

    for i, (name, act) in enumerate(zip(layer_names, activations)):
        issues = []
        if act.var() < 1e-10:
            issues.append("VERY LOW VARIANCE")
        if (np.abs(act) < dead_threshold).mean() > 0.5:
            issues.append(f"{(np.abs(act) < dead_threshold).mean()*100:.1f}% DEAD")
        if (np.abs(act) > saturated_threshold).mean() > 0.3:
            issues.append(
                f"{(np.abs(act) > saturated_threshold).mean()*100:.1f}% SATURATED"
            )
        if np.isnan(act).any():
            issues.append("CONTAINS NaN")
        if np.isinf(act).any():
            issues.append("CONTAINS Inf")

        if issues:
            print(f"  ⚠️ Layer {i} ({name}): {', '.join(issues)}")

    print("\nLayer output ranges:")
    for i, (name, act) in enumerate(zip(layer_names, activations)):
        print(
            f"  {i:3d}. {name:30s}: [{act.min():+.4f}, {act.max():+.4f}], μ={act.mean():.4f}, σ={act.std():.4f}"
        )
else:
    print("Not enough layers to plot statistics. Only have:", layer_names)

In [ ]:
# Visualize spatial activation patterns using HEALPix mollweide projections
# Show input, output, and prediction comparison


# Helper function to plot HEALPix map in a matplotlib axes using mollweide projection
def plot_healpix_map(ax, hpx_map, title, cmap="viridis", nest=True, add_colorbar=True):
    """Plot a HEALPix map on a matplotlib axes with mollweide projection.
    Does NOT remove monopole or dipole - shows raw data.
    """
    if nest:
        hpx_map = hp.reorder(hpx_map, n2r=True)

    nside = hp.npix2nside(len(hpx_map))

    # Create theta, phi grid for mollweide
    xsize = 800
    ysize = xsize // 2

    theta = np.linspace(np.pi, 0, ysize)
    phi = np.linspace(-np.pi, np.pi, xsize)
    PHI, THETA = np.meshgrid(phi, theta)

    # Convert to HEALPix pixel indices
    pix = hp.ang2pix(nside, THETA, PHI)
    grid_map = hpx_map[pix]

    # Plot on mollweide projection
    im = ax.pcolormesh(phi, np.pi / 2 - theta, grid_map, cmap=cmap, shading="auto")
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(True, alpha=0.3)

    # Add colorbar
    if add_colorbar:
        plt.colorbar(im, ax=ax, orientation="horizontal", pad=0.05, shrink=0.8)
    return im


# Create figure with mollweide subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 4), subplot_kw={"projection": "mollweide"})

# Input map
input_map = test_input[0, :, 0].numpy()
nside_in = hp.npix2nside(len(input_map))
plot_healpix_map(axes[0], input_map, f"Input (lensed)\nnside={nside_in}", cmap="RdBu_r")

# Target map
target_map = test_target[0, :, 0].numpy()
nside_out = hp.npix2nside(len(target_map))
plot_healpix_map(
    axes[1], target_map, f"Target (kappa)\nnside={nside_out}", cmap="viridis"
)

# Model prediction
prediction = u_net.predict(test_input, verbose=0)
pred_map = prediction[0, :, 0]
plot_healpix_map(
    axes[2], pred_map, f"Prediction (kappa)\nnside={nside_out}", cmap="viridis"
)

plt.suptitle("Input → Target → Prediction Comparison", fontsize=14)
plt.tight_layout()
plt.show()

# Print prediction quality metrics
print("\nPrediction Quality:")
print(f"  Target range: [{target_map.min():.4f}, {target_map.max():.4f}]")
print(f"  Prediction range: [{pred_map.min():.4f}, {pred_map.max():.4f}]")
print(f"  MSE: {np.mean((target_map - pred_map)**2):.6f}")
print(
    f"  Correlation: {np.corrcoef(target_map.flatten(), pred_map.flatten())[0,1]:.4f}"
)

# Visualize available layer activations if we have any
if len(activations) > 0:
    # Select a subset of layers to visualize (skip some if too many)
    if len(activations) > 9:
        # Sample every Nth layer plus first and last
        step = len(activations) // 8
        indices_to_plot = (
            [0] + list(range(step, len(activations) - 1, step)) + [len(activations) - 1]
        )
        indices_to_plot = sorted(set(indices_to_plot))[:9]  # Max 9
    else:
        indices_to_plot = list(range(len(activations)))

    n_plots = len(indices_to_plot)
    n_cols = min(3, n_plots)
    n_rows = (n_plots + n_cols - 1) // n_cols

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(5 * n_cols, 4 * n_rows),
        subplot_kw={"projection": "mollweide"},
    )
    if n_plots == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for ax_idx, layer_idx in enumerate(indices_to_plot):
        act = activations[layer_idx]
        name = layer_names[layer_idx]

        # Get the spatial map (first channel, first batch element)
        if len(act.shape) == 3:
            spatial_act = act[0, :, 0]  # First batch, all pixels, first channel
        elif len(act.shape) == 2:
            spatial_act = act[0, :]  # First batch, all pixels
        else:
            axes[ax_idx].set_title(f"{name}\n(cannot visualize)")
            continue

        npix = len(spatial_act)
        try:
            nside = hp.npix2nside(npix)
            plot_healpix_map(
                axes[ax_idx], spatial_act, f"{name}\n(nside={nside})", cmap="viridis"
            )
        except Exception as e:
            axes[ax_idx].set_title(f"{name}\n(npix={npix}, error)")
            print(f"Cannot visualize {name} (npix={npix}): {e}")

    # Hide unused axes
    for ax_idx in range(len(indices_to_plot), len(axes)):
        axes[ax_idx].set_visible(False)

    plt.suptitle("Layer Activation Maps (Channel 0)", fontsize=14)
    plt.tight_layout()
    plt.show()

## Transformer U-Net Model

Alternative U-Net architecture with `Healpy_Transformer` in the bottleneck for global attention.

**Key differences from ResidualHealpyUNet:**
- Bottleneck uses graph-sparse transformer attention instead of Chebyshev convolution
- Global context captured at lowest resolution (nside=4, 192 tokens)
- GELU activation in transformer; SELU for Chebyshev encoder/decoder layers
- Projection layers handle channel dimension matching

In [ ]:
# Build and compile Transformer U-Net
with strategy.scope():
    # Same learning rate schedule as SELU U-Net for fair comparison
    transformer_lr = CosineDecayRestarts(
        initial_learning_rate=5e-3,
        first_decay_steps=decay_steps,
        t_mul=2.0,
        m_mul=0.95,
        alpha=0.01,
    )

    # Transformer U-Net with GELU encoder/decoder and GELU transformer bottleneck
    transformer_unet = TransformerHealpyUNet(
        (None, core.npix, core.npols),
        activation="gelu",
        max_batch_size=batch_size,
        dropout_rate=0.1,
        # Transformer bottleneck config
        num_heads=8,
        transformer_layers=2,
        key_dim=None,  # Auto-compute to match channel dimension
    ).get_model()

    transformer_unet.compile(
        optimizer=AdamW(transformer_lr), loss="mse", metrics=["mae"]
    )

transformer_unet.summary()
print(f"\nModel parameters: {transformer_unet.count_params():,}")

In [ ]:
# Train Transformer U-Net
logger.info("Training Transformer U-Net")
transformer_history = transformer_unet.fit(
    train, epochs=10, validation_data=val, callbacks=callbacks, verbose=1
)

# Plot training comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss comparison
ax = axes[0]
if "history" in locals():
    ax.plot(history.history["loss"], "b-", label="SELU U-Net (Train)", linewidth=2)
    ax.plot(history.history["val_loss"], "b--", label="SELU U-Net (Val)", linewidth=2)
ax.plot(
    transformer_history.history["loss"],
    "r-",
    label="Transformer U-Net (Train)",
    linewidth=2,
)
ax.plot(
    transformer_history.history["val_loss"],
    "r--",
    label="Transformer U-Net (Val)",
    linewidth=2,
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (MSE)")
ax.set_title("Training Loss Comparison")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)

# Final validation loss comparison
ax = axes[1]
models = ["SELU U-Net", "Transformer U-Net"]
val_losses = [
    history.history["val_loss"][-1] if "history" in locals() else 0,
    transformer_history.history["val_loss"][-1],
]
colors = ["steelblue", "coral"]
bars = ax.bar(models, val_losses, color=colors, edgecolor="black", alpha=0.8)
ax.set_ylabel("Final Validation Loss (MSE)")
ax.set_title("Model Comparison")
ax.grid(True, alpha=0.3, axis="y")

# Add value labels on bars
for bar, val in zip(bars, val_losses):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.001,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
if "history" in locals():
    print(f"SELU U-Net final val loss:        {history.history['val_loss'][-1]:.6f}")
print(
    f"Transformer U-Net final val loss: {transformer_history.history['val_loss'][-1]:.6f}"
)

In [ ]:
# Visual comparison: Predictions from both models
fig, axes = plt.subplots(2, 3, figsize=(15, 8), subplot_kw={"projection": "mollweide"})

# Get predictions from both models
pred_selu = u_net.predict(test_input, verbose=0)[0, :, 0] if "u_net" in dir() else None
pred_transformer = transformer_unet.predict(test_input, verbose=0)[0, :, 0]
target = test_target[0, :, 0].numpy()
input_map = test_input[0, :, 0].numpy()

# Row 1: Input, Target, SELU prediction
plot_healpix_map(
    axes[0, 0],
    input_map,
    f"Input (lensed)\nnside={hp.npix2nside(len(input_map))}",
    cmap="RdBu_r",
)
plot_healpix_map(
    axes[0, 1],
    target,
    f"Target (kappa)\nnside={hp.npix2nside(len(target))}",
    cmap="viridis",
)
if pred_selu is not None:
    plot_healpix_map(axes[0, 2], pred_selu, "SELU U-Net Prediction", cmap="viridis")
else:
    axes[0, 2].set_title("SELU U-Net\n(not trained)")

# Row 2: Transformer prediction, Residual SELU, Residual Transformer
plot_healpix_map(
    axes[1, 0], pred_transformer, "Transformer U-Net Prediction", cmap="viridis"
)

if pred_selu is not None:
    residual_selu = target - pred_selu
    plot_healpix_map(
        axes[1, 1],
        residual_selu,
        f"SELU Residual\nMSE={np.mean(residual_selu**2):.4f}",
        cmap="RdBu_r",
    )
else:
    axes[1, 1].set_title("SELU Residual\n(not available)")

residual_transformer = target - pred_transformer
plot_healpix_map(
    axes[1, 2],
    residual_transformer,
    f"Transformer Residual\nMSE={np.mean(residual_transformer**2):.4f}",
    cmap="RdBu_r",
)

plt.suptitle("Model Prediction Comparison", fontsize=14)
plt.tight_layout()
plt.show()

# Print metrics
print("\nPrediction Metrics:")
print(f"  Target range: [{target.min():.4f}, {target.max():.4f}]")
if pred_selu is not None:
    print(
        f"  SELU U-Net:        MSE={np.mean((target - pred_selu)**2):.6f}, Corr={np.corrcoef(target.flatten(), pred_selu.flatten())[0,1]:.4f}"
    )
print(
    f"  Transformer U-Net: MSE={np.mean((target - pred_transformer)**2):.6f}, Corr={np.corrcoef(target.flatten(), pred_transformer.flatten())[0,1]:.4f}"
)

In [ ]:
# Visualize Transformer U-Net activations
print("Extracting Transformer U-Net activations...")

transformer_layer_names = []
transformer_activations = []

x = test_input
transformer_skips = []

for layer in transformer_unet.layers:
    if isinstance(layer, tf.keras.layers.InputLayer):
        transformer_layer_names.append("input")
        transformer_activations.append(x.numpy() if hasattr(x, "numpy") else x)
        continue

    layer_name = layer.name

    # Handle EncoderBlock (returns tuple: output, skip)
    if "encoder_block" in layer_name.lower():
        x, skip = layer(x, training=False)
        transformer_skips.append(skip)
        transformer_layer_names.append(f"{layer_name}_out")
        transformer_activations.append(x.numpy())
        transformer_layer_names.append(f"{layer_name}_skip")
        transformer_activations.append(skip.numpy())

    # Handle DecoderBlock (needs skip connection)
    elif "decoder_block" in layer_name.lower():
        if len(transformer_skips) > 0:
            skip = transformer_skips.pop()
            x = layer(x, skip=skip, training=False)
        else:
            x = layer(x, skip=None, training=False)
        transformer_layer_names.append(layer_name)
        transformer_activations.append(x.numpy())

    # Handle standard layers (HealpyGCNN bottleneck, kappa head)
    else:
        try:
            x = layer(x, training=False)
            transformer_layer_names.append(layer_name)
            if hasattr(x, "numpy"):
                transformer_activations.append(x.numpy())
            else:
                transformer_activations.append(np.array(x))
        except Exception as e:
            print(f"  Skipping {layer_name}: {e}")

print(f"\nFound {len(transformer_activations)} layer activations:")
for name, act in zip(transformer_layer_names, transformer_activations):
    print(
        f"  {name}: shape={act.shape}, min={act.min():.4f}, max={act.max():.4f}, mean={act.mean():.4f}"
    )

# Visualize transformer activations spatially
if len(transformer_activations) > 0:
    if len(transformer_activations) > 9:
        step = len(transformer_activations) // 8
        indices_to_plot = (
            [0]
            + list(range(step, len(transformer_activations) - 1, step))
            + [len(transformer_activations) - 1]
        )
        indices_to_plot = sorted(set(indices_to_plot))[:9]
    else:
        indices_to_plot = list(range(len(transformer_activations)))

    n_plots = len(indices_to_plot)
    n_cols = min(3, n_plots)
    n_rows = (n_plots + n_cols - 1) // n_cols

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        subplot_kw={"projection": "mollweide"},
    )
    if n_plots == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for ax_idx, layer_idx in enumerate(indices_to_plot):
        act = transformer_activations[layer_idx]
        name = transformer_layer_names[layer_idx]

        if len(act.shape) == 3:
            spatial_act = act[0, :, 0]
        elif len(act.shape) == 2:
            spatial_act = act[0, :]
        else:
            axes[ax_idx].set_title(f"{name}\n(cannot visualize)")
            continue

        npix = len(spatial_act)
        try:
            nside = hp.npix2nside(npix)
            plot_healpix_map(
                axes[ax_idx], spatial_act, f"{name}\n(nside={nside})", cmap="viridis"
            )
        except Exception as e:
            axes[ax_idx].set_title(f"{name}\n(npix={npix}, error)")
            print(f"Cannot visualize {name} (npix={npix}): {e}")

    for ax_idx in range(len(indices_to_plot), len(axes)):
        axes[ax_idx].set_visible(False)

    plt.suptitle("Transformer U-Net Layer Activation Maps (Channel 0)", fontsize=14)
    plt.tight_layout()
    plt.show()